# Are.na API → Webpage

This notebook pulls blocks from a public Are.na channel and generates a standalone HTML page you can download and add to your site.

**What you need:**
- The slug of a public Are.na channel (the last part of the URL, e.g. `relics-uikeqpbrexa`)

**What you'll get:**
- An HTML file that displays your blocks as a grid with images, text, links, and descriptions

No API key is required for public channels.

---
## Step 1: Set your channel slug

Go to your Are.na channel. The slug is the last part of the URL.

For example, if your channel URL is:
```
https://www.are.na/alexa-ann-bonomo/relics-uikeqpbrexa
```
Then your slug is `relics-uikeqpbrexa`.

Replace the slug below with your own:

In [ ]:
# =============================================
#  CHANGE THIS to your channel slug
# =============================================
CHANNEL_SLUG = 'relics-uikeqpbrexa'

# Optional: change the page title
PAGE_TITLE = 'relics'

---
## Step 2: Fetch blocks from Are.na

This cell calls the Are.na API and retrieves all the blocks in your channel.

The API endpoint is:
```
https://api.are.na/v2/channels/{slug}?per=100
```

Each block has a `class` that tells us what kind of content it is:
- `Image` — an uploaded image
- `Link` — a saved URL (often with a thumbnail)
- `Text` — plain text or markdown
- `Media` — embedded video/audio
- `Attachment` — an uploaded file
- `Channel` — a connected channel

In [ ]:
import requests
import json

url = f'https://api.are.na/v2/channels/{CHANNEL_SLUG}?per=100'
response = requests.get(url)

if response.status_code != 200:
    print(f'Error: API returned {response.status_code}')
    print('Make sure the channel is public and the slug is correct.')
else:
    data = response.json()
    blocks = [b for b in data.get('contents', []) if b.get('class') != 'Channel']
    print(f'Channel: {data.get("title", "untitled")}')
    print(f'Blocks found: {len(blocks)}')
    print()
    for b in blocks[:5]:
        print(f'  [{b["class"]}] {b.get("title", b.get("generated_title", "(no title)"))}')
    if len(blocks) > 5:
        print(f'  ... and {len(blocks) - 5} more')

---
## Step 3: Look at a single block

Let's inspect one block to understand the data structure.
This is useful when you want to customize what information you display.

In [ ]:
if blocks:
    print(json.dumps(blocks[0], indent=2, default=str))
else:
    print('No blocks to inspect.')

---
## Step 4: Generate the HTML page

This cell builds an HTML file from your blocks. The page includes:
- A header with your channel title
- A grid of blocks showing images, text, links, and descriptions
- Responsive layout that works on desktop and mobile

**Customization ideas** (try editing the CSS or HTML below):
- Change the grid layout (`grid-template-columns`)
- Add or remove block fields (title, description, source URL)
- Change how text blocks are displayed

In [ ]:
def block_to_html(block):
    """Convert a single Are.na block to an HTML card."""
    cls = block.get('class', '')
    title = block.get('title') or block.get('generated_title') or ''
    description = block.get('description') or ''
    source_url = ''
    if block.get('source') and block['source'].get('url'):
        source_url = block['source']['url']

    # --- visual area ---
    visual = ''
    img = block.get('image')
    if img and img.get('display') and img['display'].get('url'):
        img_url = img['display']['url']
        visual = f'<img class="block-image" src="{img_url}" alt="{title}" loading="lazy" />'
    elif cls == 'Text' and block.get('content'):
        content = block['content'][:280]
        if len(block['content']) > 280:
            content += '...'
        visual = f'<div class="block-text-content">{content}</div>'
    else:
        visual = f'<div class="block-placeholder">{cls or "block"}</div>'

    # --- title ---
    title_html = ''
    if title:
        if source_url:
            title_html = f'<div class="block-title"><a href="{source_url}" target="_blank">{title}</a></div>'
        else:
            title_html = f'<div class="block-title">{title}</div>'

    # --- source ---
    source_html = ''
    if source_url:
        try:
            from urllib.parse import urlparse
            domain = urlparse(source_url).hostname.replace('www.', '')
            source_html = f'<div class="block-source"><a href="{source_url}" target="_blank">{domain}</a></div>'
        except:
            pass

    # --- description ---
    desc_html = ''
    if description:
        short = description[:140]
        if len(description) > 140:
            short += '...'
        desc_html = f'<div class="block-description">{short}</div>'

    return f'''
    <div class="block">
      {visual}
      <div class="block-info">
        {title_html}
        {source_html}
        {desc_html}
      </div>
    </div>'''


blocks_html = '\n'.join(block_to_html(b) for b in blocks)

html = f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{PAGE_TITLE}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}

    body {{
      font-family: "Times New Roman", Times, serif;
      background: white;
      color: black;
    }}

    a {{ color: blue; }}
    a:visited {{ color: purple; }}

    header {{
      padding: 40px 20px 20px;
      max-width: 800px;
      margin: 0 auto;
    }}

    header h1 {{
      font-size: 1.8rem;
      font-weight: bold;
    }}

    header .meta {{
      margin-top: 8px;
      font-size: 0.9rem;
    }}

    .grid {{
      max-width: 800px;
      margin: 0 auto;
      padding: 20px;
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));
      gap: 20px;
    }}

    .block {{
      border: 1px solid black;
    }}

    .block-image {{
      width: 100%;
      aspect-ratio: 1 / 1;
      object-fit: cover;
      display: block;
    }}

    .block-placeholder {{
      width: 100%;
      aspect-ratio: 1 / 1;
      background: #eee;
      display: flex;
      align-items: center;
      justify-content: center;
      font-size: 0.8rem;
      color: #666;
    }}

    .block-text-content {{
      width: 100%;
      aspect-ratio: 1 / 1;
      padding: 12px;
      background: black;
      color: white;
      font-size: 0.9rem;
      line-height: 1.4;
      overflow: hidden;
    }}

    .block-info {{ padding: 10px 12px 14px; }}

    .block-title {{
      font-size: 0.85rem;
      font-weight: bold;
      line-height: 1.3;
    }}

    .block-source {{
      margin-top: 4px;
      font-size: 0.75rem;
      color: #666;
      overflow: hidden;
      text-overflow: ellipsis;
      white-space: nowrap;
    }}

    .block-description {{
      margin-top: 6px;
      font-size: 0.85rem;
      line-height: 1.4;
      color: #444;
      font-style: italic;
    }}

    @media (max-width: 600px) {{
      header {{ padding: 24px 16px 16px; }}
      .grid {{
        padding: 8px 16px 60px;
        grid-template-columns: 1fr 1fr;
        gap: 12px;
      }}
    }}
  </style>
</head>
<body>
  <header>
    <h1>{PAGE_TITLE}</h1>
    <div class="meta">
      from <a href="https://www.are.na/channel/{CHANNEL_SLUG}" target="_blank">are.na</a>
    </div>
  </header>
  <div class="grid">
{blocks_html}
  </div>
</body>
</html>'''

print(f'Generated {len(html)} characters of HTML with {len(blocks)} blocks.')

---
## Step 5: Save and download

This saves the HTML file and gives you a download link.
Add this file to your archive site.

In [ ]:
filename = f'{CHANNEL_SLUG}.html'
with open(filename, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Saved: {filename}')
print()

try:
    from google.colab import files
    files.download(filename)
    print('Download started.')
except ImportError:
    print(f'Not running in Colab. File saved to: {filename}')

---
## Step 6 (optional): Preview in the notebook

This renders a preview of the page right here in Colab.

In [ ]:
from IPython.display import HTML, display

display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" width="100%" height="600" style="border:1px solid #ccc;"></iframe>'))

---
## What to do next

1. **Edit the CSS** in Step 4 to match your site's style.
2. **Add metadata fields** to each block card: try displaying `block['connected_at']`, `block['updated_at']`, or `block['user']['username']`.
3. **Write descriptions** for your blocks on Are.na, then re-run this notebook to pull them in.
4. **Document absence**: for blocks that link to dead URLs or missing content, add descriptions on Are.na noting what is lost and what remains.
5. **Add this page to your archive site** alongside your other HTML pages.